# PARENT-CHILD RAG Pipeline
### PDF → Parent & Child Chunks → Embed Children → FAISS → Match Children → Expand to Parents → Answer

In [ ]:
!pip install langchain langchain-community langchain-ollama langchain-text-splitters faiss-cpu pypdf -q

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama
import uuid

c:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load PDF

In [2]:
loader = PyPDFLoader("A._P._J._Abdul_Kalam.pdf")
pages = loader.load()

print(f"Total Pages: {len(pages)}")
print(f"Sample Content:\n{pages[0].page_content[:500]}")

Total Pages: 30
Sample Content:
A. P. J. Abdul Kalam
Official portrait, c. 2002
President of India
In office
25 July 2002 – 25 July 2007
Prime Minister Atal Bihari Vajpayee
Manmohan Singh
Vice President Krishan Kant
Bhairon Singh Shekhawat
Preceded by K. R. Narayanan
Succeeded by Pratibha Patil
Principal Scientific Adviser to the
Government of India
In office
November 1999 – November 2001
President K. R. Narayanan
Prime Minister Atal Bihari Vajpayee
Preceded by Office established
Succeeded by Rajagopala Chidambaram
Director Ge


## Step 2: Create Parent Chunks (Large) & Child Chunks (Small)

In [3]:
# Parent chunks: large (1500 tokens)
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
parent_chunks = parent_splitter.split_documents(pages)

# Assign unique IDs to each parent
parent_store = {}
for i, parent in enumerate(parent_chunks):
    parent_id = str(i)
    parent.metadata["parent_id"] = parent_id
    parent_store[parent_id] = parent

# Child chunks: small (300 tokens), split from each parent
child_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

child_chunks = []
for parent in parent_chunks:
    children = child_splitter.split_documents([parent])
    for child in children:
        child.metadata["parent_id"] = parent.metadata["parent_id"]
    child_chunks.extend(children)

print(f"Parent Chunks: {len(parent_chunks)}")
print(f"Child Chunks: {len(child_chunks)}")
print(f"\nSample Parent (ID: 0):\n{parent_store['0'].page_content[:300]}")
print(f"\nSample Child (Parent ID: {child_chunks[0].metadata['parent_id']}):\n{child_chunks[0].page_content}")

Parent Chunks: 92
Child Chunks: 481

Sample Parent (ID: 0):
A. P. J. Abdul Kalam
Official portrait, c. 2002
President of India
In office
25 July 2002 – 25 July 2007
Prime Minister Atal Bihari Vajpayee
Manmohan Singh
Vice President Krishan Kant
Bhairon Singh Shekhawat
Preceded by K. R. Narayanan
Succeeded by Pratibha Patil
Principal Scientific Adviser to the


Sample Child (Parent ID: 0):
A. P. J. Abdul Kalam
Official portrait, c. 2002
President of India
In office
25 July 2002 – 25 July 2007
Prime Minister Atal Bihari Vajpayee
Manmohan Singh
Vice President Krishan Kant
Bhairon Singh Shekhawat
Preceded by K. R. Narayanan
Succeeded by Pratibha Patil
Principal Scientific Adviser to the


## Step 3: Embed Child Chunks & Store in FAISS

In [4]:
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

# Only child chunks are embedded
vector_store = FAISS.from_documents(child_chunks, embeddings)

print(f"Child Vector Store Created with {vector_store.index.ntotal} vectors")
print(f"Parent Doc Store has {len(parent_store)} parent chunks (stored, not embedded)")

Child Vector Store Created with 481 vectors
Parent Doc Store has 92 parent chunks (stored, not embedded)


## Step 4: Retrieve Matched Children

In [5]:
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

query = "What role did Abdul Kalam play in India's missile program?"

matched_children = retriever.invoke(query)

print(f"Matched Children: {len(matched_children)}")
for i, doc in enumerate(matched_children):
    print(f"\n--- Child {i+1} (Parent ID: {doc.metadata['parent_id']}, Page {doc.metadata['page'] + 1}) ---")
    print(doc.page_content[:200])

Matched Children: 5

--- Child 1 (Parent ID: 0, Page 1) ---
Defence Research and Development Organisation
(DRDO) and Indian Space Research Organisation
(ISRO) and was intimately involved in India's civilian
space programme and military missile development
effo

--- Child 2 (Parent ID: 6, Page 3) ---
aerospace projects under Kalam's directorship through her discretionary powers. Kalam also played a
Career as a scientist

--- Child 3 (Parent ID: 0, Page 1) ---
for his work on the development of ballistic missile
and launch vehicle technology. He also played a
pivotal organisational, technical, and political role in
Pokhran-II nuclear tests in 1998, India's 

--- Child 4 (Parent ID: 7, Page 4) ---
of missiles including Agni, an intermediate range ballistic missile and Prithvi, the tactical surface-to-
surface missile, despite inflated costs and time overruns.[28][29] He was known as the "Missil

--- Child 5 (Parent ID: 7, Page 4) ---
India" for his work on the development of ballistic mis

## Step 5: Expand to Parent Chunks

In [6]:
# Look up unique parent IDs from matched children
seen_parent_ids = set()
parent_docs = []

for child in matched_children:
    parent_id = child.metadata["parent_id"]
    if parent_id not in seen_parent_ids:
        seen_parent_ids.add(parent_id)
        parent_docs.append(parent_store[parent_id])

print(f"Expanded to {len(parent_docs)} unique parent chunks")
for i, doc in enumerate(parent_docs):
    print(f"\n--- Parent {i+1} (ID: {doc.metadata['parent_id']}, Page {doc.metadata['page'] + 1}) ---")
    print(doc.page_content[:300])

Expanded to 3 unique parent chunks

--- Parent 1 (ID: 0, Page 1) ---
A. P. J. Abdul Kalam
Official portrait, c. 2002
President of India
In office
25 July 2002 – 25 July 2007
Prime Minister Atal Bihari Vajpayee
Manmohan Singh
Vice President Krishan Kant
Bhairon Singh Shekhawat
Preceded by K. R. Narayanan
Succeeded by Pratibha Patil
Principal Scientific Adviser to the


--- Parent 2 (ID: 6, Page 3) ---
effort to develop the SLV-3 and Polar Satellite Launch Vehicle (PSLV), both of which were
successful.[24][25]
In May 1974, Kalam was invited by Raja Ramanna to witness the country's first nuclear test Smiling
Buddha as the representative of Terminal Ballistics Research Laboratory, even though he was

--- Parent 3 (ID: 7, Page 4) ---
Kalam greeting then prime minister
Vajpayee on 25 December 2003
major role in convincing the cabinet to conceal the true nature of these classified projects. His research
and leadership brought him recognition in the 1980s, which prompted the government to init

## Step 6: Setup LLM & Query with Parent Context

In [7]:
llm = ChatOllama(model="llama3.2:3b", temperature=0)

# Build context from parent chunks (full sections)
context = "\n\n".join([doc.page_content for doc in parent_docs])

# Create prompt
prompt = f"""Answer the question based only on the following context:

{context}

Question: {query}
Answer:"""

# Get response
response = llm.invoke(prompt)
print("Answer:", response.content)

Answer: According to the text, Abdul Kalam played a major role in the development of India's missile program, including:

* Developing ballistic missiles using technology from the successful SLV programme
* Directing two projects, Project Devil and Project Valiant, which sought to develop ballistic missiles
* Convincing the cabinet to conceal the true nature of these classified projects
* Playing a key role in the development of missiles, including Agni and Prithvi, despite inflated costs and time overruns
* Being known as the "Missile Man of India" for his work on the development of ballistic missile and launch vehicle technology.


## Parent Source Chunks

In [8]:
for i, doc in enumerate(parent_docs):
    print(f"\n--- Chunk {i+1} (Page {doc.metadata['page'] + 1}) ---")
    print(doc.page_content[:300])


--- Chunk 1 (Page 1) ---
A. P. J. Abdul Kalam
Official portrait, c. 2002
President of India
In office
25 July 2002 – 25 July 2007
Prime Minister Atal Bihari Vajpayee
Manmohan Singh
Vice President Krishan Kant
Bhairon Singh Shekhawat
Preceded by K. R. Narayanan
Succeeded by Pratibha Patil
Principal Scientific Adviser to the


--- Chunk 2 (Page 3) ---
effort to develop the SLV-3 and Polar Satellite Launch Vehicle (PSLV), both of which were
successful.[24][25]
In May 1974, Kalam was invited by Raja Ramanna to witness the country's first nuclear test Smiling
Buddha as the representative of Terminal Ballistics Research Laboratory, even though he was

--- Chunk 3 (Page 4) ---
Kalam greeting then prime minister
Vajpayee on 25 December 2003
major role in convincing the cabinet to conceal the true nature of these classified projects. His research
and leadership brought him recognition in the 1980s, which prompted the government to initiate an
advanced missile programme unde


Test Questions

1. What year was Abdul Kalam born?
2. What was the name of the coronary stent Kalam co-developed?
3. Which institution did Kalam study aerospace engineering at?
4. How many electoral votes did Kalam receive in the 2002 presidential election?
5. What was Kalam's role at DRDO from 1992 to 1999?